# Data Source Extraction

This notebook explains the output shape for data-source extraction and shows lightweight checks for reviewing extracted items.


## Review The Task Configuration

`FindDataSourcesConfig` defines the retrieval templates and prompt used by `PrecisionMiner` for data-source extraction.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import tempfile


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "episcope").exists():
            return candidate
    return None


PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is not None:
    src_path = str(PROJECT_ROOT / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

WORK_DIR = Path(tempfile.mkdtemp(prefix="episcope-notebook-"))
WORK_DIR


In [ ]:
from episcope.workflows.precision_miner import FindDataSourcesConfig

config = FindDataSourcesConfig(top_k=5)
print("top_k:", config.top_k)
print("retrieval templates:")
for template in config.retrieval_templates[:4]:
    print("-", template)


## Validate Generator JSON

The parser accepts a single JSON object with a task-level description and an `items` list. Each item should identify the source, explain why it matters, and preserve the raw supporting text when possible.


In [ ]:
import json

from episcope.workflows.precision_miner.parsing import PrecisionMinerResponseParser

raw_response = json.dumps(
    {
        "description": "The study uses registry data and a public repository.",
        "items": [
            {
                "name": "National Hospital Registry",
                "url": None,
                "explanation": "Named as the source of patient admissions and outcomes.",
                "raw_text": "The analysis used patient records from the National Hospital Registry.",
            },
            {
                "name": "Public analysis repository",
                "url": "https://example.org/repository",
                "explanation": "Hosts de-identified trial data and code.",
                "raw_text": "De-identified trial data and the analysis code are available from the public repository.",
            },
        ],
    }
)

parsed = PrecisionMinerResponseParser.parse(raw_response)
parsed.model_dump()


## Review Evidence Coverage

A useful extraction should be tied back to retrieved chunks. The review table below is a simple pattern for checking whether each item has enough support.


In [ ]:
from episcope.schemas import SearchResult

candidate_evidence = [
    SearchResult(
        id="chunk-1",
        paper_id="paper_open_data",
        section_title="Methods",
        section_type="Methods",
        text="The analysis used patient records from the National Hospital Registry.",
        similarity_score=0.91,
    ),
    SearchResult(
        id="chunk-2",
        paper_id="paper_open_data",
        section_title="Data availability",
        section_type="Data availability",
        text="De-identified trial data and the analysis code are available from the public repository.",
        similarity_score=0.86,
    ),
]

review_rows = []
for item in parsed.items:
    supporting = [chunk for chunk in candidate_evidence if item.raw_text and item.raw_text[:40] in chunk.text]
    review_rows.append(
        {
            "name": item.name,
            "has_url": bool(item.url),
            "supporting_chunks": len(supporting),
            "best_section": supporting[0].section_title if supporting else None,
        }
    )

review_rows


## Use The Output

Keep the compact `ExtractionResult` for downstream tables and dashboards. Keep the detailed workflow trace during review so uncertain items can be traced back to prompt messages, raw generator output, and supporting chunks.
